In [1]:
import pandas as pd

train_df = pd.read_csv("../data/train.csv")

print(train_df.shape)

train_df.head()

(105000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,0,0.000000,49,0,0.127227,5163.0,5,0,0,0,0.0
1,0,0.033105,37,0,0.423063,2800.0,4,0,1,0,1.0
2,0,0.007948,39,0,0.685663,3333.0,11,0,1,0,0.0
3,0,0.576297,57,0,0.320077,15161.0,14,0,4,0,0.0
4,0,0.623724,35,0,0.534226,6500.0,9,0,3,0,1.0


In [2]:
train_df.isnull().sum()

SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           20723
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       2721
dtype: int64

In [4]:
# Handle missing values using median imputation

train_df["MonthlyIncome"] = train_df["MonthlyIncome"].fillna(
    train_df["MonthlyIncome"].median()
)

train_df["NumberOfDependents"] = train_df["NumberOfDependents"].fillna(
    train_df["NumberOfDependents"].median()
)

In [5]:
# Verify that all missing values have been handled

train_df.isnull().sum()

SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64

In [6]:
# Separate independent features and target variable

X = train_df.drop("SeriousDlqin2yrs", axis=1)

y = train_df["SeriousDlqin2yrs"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (105000, 10)
y Shape: (105000,)


In [7]:
# Check target class distribution

print(y.value_counts())

print("\nPercentage Distribution:")

print(y.value_counts(normalize=True))

SeriousDlqin2yrs
0    97982
1     7018
Name: count, dtype: int64

Percentage Distribution:
SeriousDlqin2yrs
0    0.933162
1    0.066838
Name: proportion, dtype: float64


The target variable is highly imbalanced, with approximately 93.3% non-default customers and 6.7% default customers. This is a common characteristic of real-world credit risk datasets.

In [8]:
# Split training data into train and test sets

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)

X_train Shape: (84000, 10)
X_test Shape: (21000, 10)


In [9]:
# Train Logistic Regression credit risk model

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Training Completed")

Model Training Completed


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
# Generate predictions and probability scores

y_pred = model.predict(X_test)

y_pred_prob = model.predict_proba(X_test)[:,1]

In [11]:
# Evaluate model performance using ROC-AUC

from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_pred_prob)

print("AUC Score:", round(auc,4))

AUC Score: 0.6879


In [12]:
# Calculate classification performance metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall   :", round(recall,4))
print("F1 Score :", round(f1,4))

Accuracy : 0.9342
Precision: 0.5852
Recall   : 0.0563
F1 Score : 0.1027


In [13]:
# Calculate Gini coefficient

gini = 2 * auc - 1

print("Gini Coefficient:", round(gini,4))

Gini Coefficient: 0.3758


In [14]:
# Calculate KS statistic

from scipy.stats import ks_2samp

ks_statistic = ks_2samp(
    y_pred_prob[y_test == 0],
    y_pred_prob[y_test == 1]
)

print("KS Statistic:", round(ks_statistic.statistic,4))

KS Statistic: 0.3093


In [15]:
# Create model performance summary table

import pandas as pd

metrics_df = pd.DataFrame({
    "Metric": [
        "AUC",
        "Gini",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "KS Statistic"
    ],
    "Value": [
        auc,
        gini,
        accuracy,
        precision,
        recall,
        f1,
        ks_statistic.statistic
    ]
})

metrics_df

,Metric,Value
0,AUC,0.687904
1,Gini,0.375808
2,Accuracy,0.934238
3,Precision,0.585185
4,Recall,0.056268
5,F1 Score,0.102664
6,KS Statistic,0.309256


In [16]:
# Save trained model for validation framework

import joblib

joblib.dump(model, "../models/credit_model.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [17]:
# Verify saved model file

import os

print(os.listdir("../models"))

['credit_model.pkl']


In [18]:
# Save model performance metrics

metrics_df.to_csv(
    "../models/model_metrics.csv",
    index=False
)

print("Metrics Saved Successfully")

Metrics Saved Successfully


In [19]:
# Verify all saved files

import os

print(os.listdir("../models"))

['model_metrics.csv', 'credit_model.pkl']
